# Lab 4 — Forecasting: How Many Riders Next Week?  🚆📈
### AI & Data Science with GenAI (Rail) · Advanced ML — your forecasting lab

In **Lab 2** you built a model that looked at *today's* sensors and said *"this is a failure."*
That was **detection**. Today you do something genuinely harder: you predict a number for a
day that **has not happened yet**.

**The scenario.** You sit in the operations planning team of the **Long Island Rail Road
(LIRR)** — one of the busiest commuter railroads in the world. Every week you must publish
**next week's train and crew plan**. To do that you need a daily ridership forecast for the
**seven days ahead**, so the planners know which days need extra cars and which need fewer.

**The data is real.** Daily ridership counts published by the **MTA**, from January 2023 to
August 2026 — 1,339 days. Nothing is invented.

> **The rule that shapes this whole lab.** You publish the plan **a week in advance**. So when
> you forecast next Tuesday, you do **not** yet know Monday's numbers — you do not know
> *anything* from the last 7 days. Every feature we build has to be at least **7 days old**.
> Break that rule and you build a model that looks brilliant in the notebook and is useless
> in the depot.

**What you will do:** clean the series → understand its rhythm → split by time → build three
dumb baselines → score them properly → train a regression → and then find out whether the
regression was worth the trouble.

**Your deliverable is a written report** (`Student_Report_Lab4.docx`), in your own words.

## Step 0 — Get our tools ready and load the data
Same libraries as Lab 2. From scikit-learn we take **`LinearRegression`** (you met regression
in the Regression Models session) and the **error metrics** we will score forecasts with.

In [ ]:
# ============================================================
# STEP 0 : import our tools and load the dataset
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# The machine-learning toolkit, same as Lab 2 -- but a different model this time.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# One row = ONE DAY of ridership on the MTA network.
#   lirr_riders    = Long Island Rail Road    <- the series we forecast
#   mnr_riders     = Metro-North Railroad     <- a second commuter railroad (for the Challenge)
#   subway_riders  = New York City Subway     <- kept so you can sanity-check odd days
df = pd.read_csv("lirr_daily_ridership.csv", parse_dates=["date"])

# ALWAYS sort a time series by time before doing anything else.
df = df.sort_values("date").reset_index(drop=True)

TARGET = "lirr_riders"          # the number we are trying to forecast
df.head()

## Step 1 — First look at the series
With a time series the first question is never "what is the mean" — it is **"what shape does
it make?"** So we plot it.

In [ ]:
print("Rows and columns:", df.shape)          # 1339 days, 4 columns
print("From", df["date"].min().date(), "to", df["date"].max().date())

# A time series with a MISSING DAY is dangerous: "shift by 7 rows" would stop meaning
# "shift by 7 days". So check the calendar is complete before we rely on that.
expected_days = len(pd.date_range(df["date"].min(), df["date"].max()))
print("Days expected:", expected_days, "| rows we have:", len(df), "| missing days:", expected_days - len(df))

plt.figure(figsize=(13, 4))
plt.plot(df["date"], df[TARGET], lw=0.7)
plt.title("LIRR daily ridership, Jan 2023 - Aug 2026")
plt.xlabel("date"); plt.ylabel("riders per day")
plt.tight_layout(); plt.show()

> **Reading it:** two things are going on at once. There is a slow **upward drift** — the
> railroad is still recovering ridership year on year. And the line is a **thick fuzzy band**
> rather than a thin curve: that thickness is a pattern repeating every few days, far too fast
> to see at this zoom. We go find it in Step 4.
>
> **Zero missing days** — good. That means shifting the table by 7 rows is exactly the same as
> going back 7 days, which is what the whole lab depends on.

In [ ]:
# 🔧 Your turn (think, don't copy):
# Zoom in on ONE MONTH so you can actually see the shape - say June 2026.
# Approach: build a smaller frame with a date filter, exactly like Lab 2 Step 3
#   (df["date"] >= "2026-06-01") & (df["date"] <= "2026-06-30")
# then plot date against lirr_riders on that frame.
# Then, in a comment: how many peaks do you count in the month, and roughly
# how many days apart are they?




## Step 2 — Data preparation: find the impossible days  ⭐ new idea
Before any model, look for values that **cannot be true**. On a railway that carries a
quarter of a million people a day, a count of **zero** is not a quiet day — it is a
**reporting failure**. And when you find one, the other columns tell you what kind.

In [ ]:
print(df[TARGET].describe().round(0))

# The minimum is 0. Which days - and what were the OTHER two railways doing?
bad_days = df[df[TARGET] == 0]
print("\nDays reporting ZERO LIRR riders:")
print(bad_days.to_string(index=False))

> **Reading it:** four days report zero LIRR riders — but they are **not all the same kind of
> day**, and the other two columns are what tell you so.
>
> - **16, 17 and 18 May 2026:** Metro-North and the subway carried *above*-normal loads. The
>   network was busy. A railway that is running does not carry literally nobody, so this is a
>   **gap in the LIRR feed**.
> - **23 February 2026:** everything collapsed together — Metro-North at about 8% of a normal
>   Monday, the subway at about 24%. Something shut the whole network down that day. The LIRR
>   figure is *still* impossible (zero, not merely low), so we still repair it — but we should
>   **flag that repair as a guess**, not treat it as recovered fact.
>
> **How to fix a gap in a time series.** Do **not** delete the rows — that would break the
> "7 rows = 7 days" rule we just checked. Do **not** use the overall average either: a Sunday
> filled with the all-days average would be far too high. Fill each gap with **the same weekday
> from the week before** — the closest honest guess we have.

In [ ]:
# Step 1: turn the impossible zeros into "missing" so pandas stops treating them as data.
df[TARGET] = df[TARGET].replace(0, np.nan)

# Step 2: fill each gap with the value 7 rows earlier = the same weekday, one week back.
df[TARGET] = df[TARGET].fillna(df[TARGET].shift(7))

print("Values now sitting on those four dates:")
print(df[df["date"].isin(bad_days["date"])][["date", TARGET]].to_string(index=False))
print("\nMissing values left:", int(df[TARGET].isna().sum()))
print("New minimum:", int(df[TARGET].min()))

> **One more thing an honest analyst writes down.** The 23 February gap was filled from
> **16 February 2026 — Presidents' Day**, a federal holiday, and therefore an unusually quiet
> Monday itself. So that single repaired value is doubly shaky: it stands in for a day the
> network was shut, using a number from a day the network was on holiday. It is one row out of
> 1,339 and it sits in the training period, so it will not move our results — but "it does not
> matter much" is a conclusion you are only allowed to state **after** you have noticed it.

In [ ]:
# 🔧 Your turn:
# The new minimum is about 30,577 riders - still very low for this railroad.
# Find WHICH DATE that is, and what day of the week it fell on.
# Approach: the row you want is the one where lirr_riders equals the column's .min().
#   A date column has a .dt accessor: df["date"].dt.day_name() gives "Monday", "Tuesday"...
# Then, in a comment: use the same reasoning we just used on the four zeros. Is this a
# data error, or a real (very quiet) day? What would you need to check to be sure?




## Step 3 — Split into TRAIN and TEST — by time  ⭐
Same rule as Lab 2, and it matters even more here: **train on the past, test on the future.**
Never shuffle a time series. If a single future day leaks into training, your scores become
fiction.

We cut at **1 June 2026**, which leaves the last **13 weeks** (a full quarter) as an honest
test of "could this have planned the summer?".

We do this **now, before we explore the data**, on purpose. Everything we measure or build
from here on uses the training period only; the test set is not scored until Step 6, and
nothing is ever *fitted* on it.

In [ ]:
CUTOFF = pd.Timestamp("2026-06-01")

train = df[df["date"] <  CUTOFF]
test  = df[df["date"] >= CUTOFF]

print("train:", len(train), "days |", train["date"].min().date(), "->", train["date"].max().date())
print("test :", len(test),  "days |", test["date"].min().date(),  "->", test["date"].max().date())

> **Insight — be honest about what a 13-week test can and cannot tell you.** 92 days is
> enough to see whether the model has the weekly rhythm right. It is **not** enough to judge
> it across a whole year: our test is entirely **summer**, and summer on a commuter railroad
> does not look like February. A model graded only on June–August has never been asked about
> snow, or the September back-to-work surge. Say this in your report.

In [ ]:
# 🔧 Your turn:
# Prove to yourself the split does not leak. Print the LAST date in train and the FIRST
# date in test, and how many days of test data there are as a share of the whole dataset.
# Approach: .max() and .min() on the date column of each frame; a share is len(test)/len(df).
# Then, in a comment: state whether the two periods overlap (yes / no).




## Step 4 — The rhythm: what repeats?
Forecasting is mostly about finding **what repeats** and betting that it repeats again. Step 1
hinted at a weekly cycle. Let's measure it — **on the training period only**, because a number
you measure using the test set is a number you are no longer allowed to be surprised by.

In [ ]:
# These two columns describe the calendar, so they belong on the whole frame.
df["dayname"] = df["date"].dt.day_name()      # "Monday", "Tuesday", ...
df["dow"]     = df["date"].dt.dayofweek        # 0 = Monday ... 6 = Sunday

# Adding columns to df does NOT update the slices we cut in Step 3 - refresh them.
train = df[df["date"] <  CUTOFF]
test  = df[df["date"] >= CUTOFF]

order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_day = train.groupby("dayname")[TARGET].mean().reindex(order)     # TRAIN only

print(by_day.round(0).to_string())
print("\nBusiest day / quietest day ratio:", round(by_day.max() / by_day.min(), 2))

plt.figure(figsize=(9, 4))
plt.bar(by_day.index, by_day.values, color="steelblue")
plt.title("Average LIRR riders by day of the week (training period only)")
plt.ylabel("riders per day"); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

> **Insight:** a Wednesday carries **2.27 times** as many riders as a Sunday. That single
> fact dominates everything else in this dataset — the year-on-year growth is a few percent,
> the weekday effect is over a hundred percent. **Any forecast that ignores the day of the
> week is going to be badly wrong on five days out of seven.**

In [ ]:
# 🔧 Your turn:
# Is there a pattern across the YEAR as well? Work out the average ridership per calendar
# month - again on the TRAINING data only.
# Approach: train["date"].dt.month gives 1-12; group by that and take the mean of the
#   target, exactly like the groupby above.
# Then, in a comment: name the quietest month and the busiest month, and say whether this
# yearly swing is BIGGER or SMALLER than the weekday swing you just measured.




## Step 5 — Baselines BEFORE models  ⭐ the habit that saves careers
In Lab 2 you learned to compare a model against something dumb. In forecasting that is not
just good practice — it is the **standard**. A forecast is never "good" on its own; it is only
ever **better or worse than the obvious thing**.

Three obvious things:

| Baseline | The rule | Available 7 days ahead? |
|---|---|---|
| **naive** | "tomorrow = today" | ❌ **no** |
| **seasonal naive** | "next Tuesday = last Tuesday" | ✅ yes |
| **flat** | "every day = the recent average" | ✅ yes |

Look hard at that last column. The naive baseline needs **yesterday's** number, and when you
publish the plan a week early you simply do not have it. We compute it anyway — to show you
exactly how much a model gains from information it is **not allowed to use**.

In [ ]:
# Each baseline is one line, because a baseline should be.
df["naive"]  = df[TARGET].shift(1)    # yesterday               (NOT usable 7 days ahead)
df["snaive"] = df[TARGET].shift(7)    # same weekday last week  (usable)
# The 7-day rule applies to baselines too. On the day we publish, the newest count we have
# is 7 days old - so the "recent average" must stop 7 days before the first test day.
last_known = CUTOFF - pd.Timedelta(days=7)
df["flat"] = df[df["date"] <= last_known][TARGET].tail(28).mean()

print("Newest day we are allowed to use:", last_known.date())
print("The flat forecast is a single number:", round(df["flat"].iloc[0]))

# Rebuild train/test so they carry the new baseline columns.
train = df[df["date"] <  CUTOFF]
test  = df[df["date"] >= CUTOFF]

# Look at the seasonal-naive forecast against reality for the first three test weeks.
first3 = test.head(21)
plt.figure(figsize=(12, 4))
plt.plot(first3["date"], first3[TARGET],  marker="o", label="actual")
plt.plot(first3["date"], first3["snaive"], marker="x", ls="--", label="seasonal naive (last week)")
plt.title("Seasonal naive vs reality - first three weeks of the test period")
plt.ylabel("riders per day"); plt.legend(); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

> **Insight:** "the same day last week" already tracks the shape astonishingly well — the
> weekday peaks and weekend troughs line up almost perfectly. That is the bar your model has
> to clear. Notice where the dashed line *misses*: it is late reacting to anything unusual,
> because it is literally repeating last week.

In [ ]:
# 🔧 Your turn:
# The flat baseline predicts the SAME number every single day (about 221,400).
# Without computing any error yet: on which days of the week will it be far too HIGH,
# and on which far too LOW?
# Approach: compare that one number against the weekday averages you printed in Step 4.
# Then, in a comment: name the days it overshoots and the days it undershoots, and say
# what that tells you about forecasting a series with strong weekly seasonality.




## Step 6 — Scoring a forecast: MAE, RMSE, MAPE  ⭐ new idea
A forecast of a **number** is not right or wrong — it is off by some amount. Three ways to
summarise "how far off, on average":

- **MAE** — Mean Absolute Error. Average miss, **in riders**. The one you quote in a meeting.
- **RMSE** — Root Mean Squared Error. Squares the misses first, so a few **big** misses hurt
  much more than many small ones. Use it when one catastrophic day matters more than steady
  small errors.
- **MAPE** — Mean Absolute Percentage Error. The miss as a **% of the actual**. Lets you
  compare across railroads of different sizes.

There is no "accuracy" here. Accuracy is for classification — this is regression.

In [ ]:
# One function, used for every forecast in this notebook - baselines and model alike.
def scores(actual, forecast):
    actual   = np.asarray(actual,   dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    mae  = mean_absolute_error(actual, forecast)
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    mape = 100 * np.mean(np.abs((actual - forecast) / actual))
    return mae, rmse, mape

print(f"{'baseline':<16}{'MAE':>10}{'RMSE':>10}{'MAPE':>8}")
for name in ["naive", "snaive", "flat"]:
    mae, rmse, mape = scores(test[TARGET], test[name])
    print(f"{name:<16}{mae:>10,.0f}{rmse:>10,.0f}{mape:>7.2f}%")

> **Insight:** read the table carefully, because it contains the whole lesson of the lab.
>
> - **flat** is hopeless (MAE ≈ 64,100) — as you predicted, it ignores the weekly rhythm.
> - **naive** is also bad (MAE ≈ 46,000) — "tomorrow looks like today" is a terrible rule when
>   today might be a Sunday and tomorrow a Monday. And remember: you are **not even allowed
>   to use it** at a 7-day horizon.
> - **seasonal naive** wins easily (MAE ≈ 21,300) — and it is a **one-line rule with no
>   model in it at all**.
>
> Write that number down. **21,254 riders** is what your regression has to beat to have been
> worth building.

In [ ]:
# 🔧 Your turn:
# An MAE of 21,254 riders sounds enormous. Put it in proportion.
# (a) Express the seasonal-naive MAE as a PERCENTAGE of average daily ridership in the
#     test period. (test[TARGET].mean() gives you that average.)
# (b) Check your answer against the MAPE in the table above.
# Approach: percentage = 100 * MAE / average.
# Then, in a comment: the two numbers are close but not identical - suggest why
#   (hint: one divides by the average day, the other divides day by day).




## Step 7 — Build features, then train the regression  ⭐
Now the model. A regression cannot read a date — it needs **columns of numbers**. So we
manufacture them. This step is called **feature engineering**, and in forecasting it *is* the
job: the model is ordinary linear regression, exactly what you met in the Regression session.

Four kinds of feature, and one rule governing all of them — **nothing newer than 7 days**:

| Feature | What it carries | Known 7 days ahead? |
|---|---|---|
| `lag_7` | ridership on the same weekday last week | ✅ |
| `lag_14` | ridership two weeks ago | ✅ |
| `dow_1 … dow_6` | which day of the week it is | ✅ (it's a calendar) |
| `t` | how many days since the start — the slow growth trend | ✅ (it's a counter) |

`lag_7` is the newest thing we allow ourselves, and it is exactly 7 days old. That is what
makes this a **true 7-day-ahead forecast** rather than a trick.

> **A pedant's footnote, and pedants are right here.** These counts are published the *next*
> day. So if the plan for day D goes out on day D−7, the count for D−7 is not on your desk
> yet — strictly you would use `lag_8`. We checked: it costs almost nothing (MAE 18,505
> instead of 18,113). Knowing *when your data actually arrives*, not just what it contains,
> is part of the job.

In [ ]:
# --- the lag features: yesterday is banned, last week is allowed
df["lag_7"]  = df[TARGET].shift(7)     # same weekday, one week ago
df["lag_14"] = df[TARGET].shift(14)    # same weekday, two weeks ago

# --- the trend counter: 0, 1, 2, ... lets the model learn the slow year-on-year growth
df["t"] = np.arange(len(df))

# --- day of week as SEPARATE 0/1 COLUMNS (one-hot encoding).
# drop_first=True removes Monday, so every coefficient reads "compared with a Monday".
dow_dummies = pd.get_dummies(df["dow"], prefix="dow", drop_first=True).astype(int)
print("day-of-week columns:", list(dow_dummies.columns))

# --- glue it together and drop the first 14 rows, which have no lag values yet
model_df = pd.concat([df[["date", TARGET, "lag_7", "lag_14", "t"]], dow_dummies], axis=1).dropna()
FEATURES = ["lag_7", "lag_14", "t"] + list(dow_dummies.columns)

# --- split by time again, on the modelling frame
tr = model_df[model_df["date"] <  CUTOFF]
te = model_df[model_df["date"] >= CUTOFF]
print("model train rows:", len(tr), "| model test rows:", len(te))

X_train, y_train = tr[FEATURES], tr[TARGET]
X_test,  y_test  = te[FEATURES], te[TARGET]

model = LinearRegression()
model.fit(X_train, y_train)          # <-- the "learning" step
print("\nTrained on", len(X_train), "days.")

In [ ]:
# 🔧 Your turn:
# A linear regression is readable - print what it learned and translate it into English.
# Print each feature name next to its coefficient, and the intercept.
# Approach: model.coef_ lines up with FEATURES in order; model.intercept_ is a single number.
#   pd.Series(model.coef_, index=FEATURES) puts them side by side neatly.
# Then, in a comment: dow_5 and dow_6 are Saturday and Sunday. What do their coefficients
#   say, in plain words, about a weekend day compared with a Monday? And what does the
#   coefficient on t say about the railroad's ridership over time?




## Step 8 — Did the model beat the one-line rule?
Now the only comparison that matters. The regression used **nine engineered features and
1,233 days of training**. Seasonal naive used **one lookup**. Score them on the same 92 days.

In [ ]:
pred = model.predict(X_test)

mae_m, rmse_m, mape_m = scores(y_test, pred)
mae_s, rmse_s, mape_s = scores(y_test, te["lag_7"])     # lag_7 IS the seasonal-naive forecast

print(f"{'':<18}{'MAE':>10}{'RMSE':>10}{'MAPE':>8}")
print(f"{'seasonal naive':<18}{mae_s:>10,.0f}{rmse_s:>10,.0f}{mape_s:>7.2f}%")
print(f"{'our regression':<18}{mae_m:>10,.0f}{rmse_m:>10,.0f}{mape_m:>7.2f}%")
print(f"\nMAE improvement over the one-line rule: {100*(mae_s-mae_m)/mae_s:.1f}%")

plt.figure(figsize=(13, 4))
plt.plot(te["date"], y_test,        lw=1.6, label="actual")
plt.plot(te["date"], pred,          lw=1.2, label="regression forecast")
plt.plot(te["date"], te["lag_7"],   lw=1.0, ls="--", alpha=0.7, label="seasonal naive")
plt.title("Forecasting the summer of 2026 - 7 days ahead, every day")
plt.ylabel("riders per day"); plt.legend(); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

> **Insight — the honest headline.** The regression wins: MAE falls from **21,254** to
> **18,113**, about **15% better**, and RMSE improves much more (35,325 → 25,402), which tells
> you the model's real gain is that it **avoids the disasters**, not that it is slightly
> sharper every day.
>
> But hold the champagne. Fifteen percent is the *entire* return on nine engineered features
> and a trained model, against a rule you can write on a napkin. **The weekly rhythm was doing
> most of the work all along** — the model's job was only to polish it. That is the normal
> result in forecasting, and reporting it plainly is what separates an analyst from a
> salesperson.

In [ ]:
# 🔧 Your turn:
# Your manager asks: "so what does an MAE of about 18,000 riders actually mean for us?"
# (a) Express the model's MAE as a percentage of average test-period ridership.
# (b) Express it as a percentage of a typical SUNDAY (use the Step 4 weekday averages).
# Approach: 100 * mae_m / the relevant average, twice.
# Then, in a comment: answer the manager in one sentence, and say which of the two
#   percentages you would put in front of the weekend planning team, and why.




## Step 9 — Look at what it got wrong  ⭐ think like a skeptic
An average error tells you almost nothing about *when* you will be embarrassed. So stop
looking at the summary and **look at the individual days.**

In [ ]:
errors = te[["date", TARGET]].copy()
errors["forecast"]  = pred.round()
errors["error"]     = (errors[TARGET] - errors["forecast"]).round()   # + = we under-forecast
errors["abs_error"] = errors["error"].abs()
errors["dayname"]   = errors["date"].dt.day_name()

print("The 8 worst days:")
print(errors.sort_values("abs_error", ascending=False).head(8).to_string(index=False))

print("\nAverage absolute error by day of the week:")
print(errors.groupby("dayname")["abs_error"].mean().reindex(order).round(0).to_string())

> **Insight:** the errors are **not** spread evenly, and that is the most useful thing on
> this page.
>
> - The single worst day is **Friday 3 July 2026** — the observed Independence Day holiday.
>   The model forecast 270,646 and only 150,445 turned up. It was wrong by **120,201 riders**,
>   because *nothing in its nine features tells it that a holiday exists*.
> - The next-worst **over**-forecast, **Friday 19 June**, is Juneteenth. Same cause.
> - **Friday is by far the worst weekday** (MAE 34,311, versus 9,227 on Tuesday). Fridays carry
>   holidays, summer getaways and variable commuting — the model treats every Friday alike.
> - **Thursday 18 June** is the second-worst miss and the reverse problem: 369,482 riders, the
>   busiest day in the whole 1,339-day extract, and the model had no idea it was coming.
>   We do not know what — and an honest report **says so** rather than inventing a reason.
>
> Every one of these is a **missing feature**, not a broken model. That is the direction of
> travel: you improve a forecast by giving it better information, not by trying fancier maths.

In [ ]:
# 🔧 Your turn:
# Two of the three worst days were public holidays, and the model over-forecast BOTH.
# (a) Add a column that flags whether each error is an over-forecast or an under-forecast,
#     and count how many of each there are in the 92 test days.
# (b) Compute the model's BIAS: the mean of the raw error column (not the absolute one).
# Approach: an over-forecast is a row where "error" is negative. (errors["error"] < 0).sum()
#   counts them. The mean of "error" is one call to .mean().
# Then, in a comment: is the model on average too high or too low overall, and would you
#   want a forecast that is systematically biased in one direction? (Think about who is
#   hurt by each kind of mistake on a railway.)




## 🏁 Challenge (do BOTH)

**1. Teach it the calendar.** The worst error in the whole test period was a public holiday.
Add a single **holiday flag** feature, retrain, and measure how much it helps.

*Approach:* pandas already knows the US federal holiday calendar —

```python
from pandas.tseries.holiday import USFederalHolidayCalendar
hols = USFederalHolidayCalendar().holidays(start="2022-12-01", end="2026-12-31")
df["holiday"] = df["date"].isin(hols).astype(int)
```

Then rebuild `model_df` with `"holiday"` added to `FEATURES`, refit, and re-score.
Report the new MAE and MAPE, the improvement over Step 8, and what the **coefficient on
`holiday`** means in riders.

**2. Put a price on being wrong.** The two mistakes are not equally expensive. Say
**over-forecasting by one rider costs 1** (an empty seat, a little wasted crew time) but
**under-forecasting by one rider costs 3** (crowding, passengers left behind, complaints).

- Write a `cost(actual, forecast)` function using those weights and compute the total cost of
  the Step 8 model and of seasonal naive across the test period.
- Then try **deliberately adding a safety margin** to every forecast — `pred + bump` for
  `bump` in `0, 2000, 4000, ... 40000`. For each bump print **both the cost and the MAE**.
- Find the bump with the **lowest cost**, and separately the bump with the **lowest MAE**.
  They are **not in the same place**. That gap is the whole point.
- In your report, answer the question this raises: *is the most accurate forecast the same
  thing as the most useful one — and who gets to decide which one ships?*

> ⚠️ **One honesty check to include.** You are about to choose the safety margin by testing
> margins on the very 92 days you then report the saving on. That is the same sin the lab
> warned you about in Step 3, wearing a different hat. A real deployment tunes the margin on
> a **validation window inside the training period** and only then measures it on the test
> set. Try it if you have time — hold out March–May 2026, pick the cheapest bump there, and
> see what it costs on the real test period.

In [ ]:
# 🏁 Challenge - your workspace

# Part 1: add a holiday flag and re-score


# Part 2: price the mistakes, then find the cheapest safety margin
#         (print cost AND mae for each bump - they bottom out in different places)




## ✅ Now write your report
You have built a genuine **7-day-ahead** forecast, scored it against rules a child could
follow, and found out exactly where it breaks. The **deliverable is your report**
(`Student_Report_Lab4.docx`), in **your own words**. Make sure it covers:

- **What you forecast and how far ahead**, and why every feature had to be at least 7 days old;
- **The baselines** — what seasonal naive scored, and how much your regression actually added
  (quote MAE and MAPE, and say plainly whether the gain justified the work);
- **Which metric you would report to management and why** (MAE in riders? MAPE? RMSE?);
- **Where the model fails** — the holiday misses, the Friday problem, the 18 June spike you
  cannot explain — and what feature would fix each;
- **The cost argument** from the Challenge: the cheapest safety margin and the most accurate
  one are not the same margin — so is the most accurate forecast the most useful one, and who
  decides? Say whether you picked your margin honestly (on a validation window) or on the
  test set, and what that does to the saving you can claim.
- **The honest limits**: 92 test days, all of them summer; one railroad; a straight-line trend
  that will not stay true forever; and four days of data we had to repair before we started.

> Quote a number as evidence where it helps — but a report that is only pasted output scores
> no marks. Tell us what it **means for the people planning next week's trains.**